In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("./online_retail_data.csv")
print(df)

        Invoice StockCode                          Description  Quantity  \
0        489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1        489434    79323P                   PINK CHERRY LIGHTS        12   
2        489434    79323W                  WHITE CHERRY LIGHTS        12   
3        489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4        489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   
...         ...       ...                                  ...       ...   
1067366  581587     22899         CHILDREN'S APRON DOLLY GIRL          6   
1067367  581587     23254        CHILDRENS CUTLERY DOLLY GIRL          4   
1067368  581587     23255      CHILDRENS CUTLERY CIRCUS PARADE         4   
1067369  581587     22138        BAKING SET 9 PIECE RETROSPOT          3   
1067370  581587      POST                              POSTAGE         1   

                 InvoiceDate  Price  Customer ID         Country  
0        2009-12-01 

In [3]:
# Remove additional characters and converted CustomerID from float to char so that the DB saves space
# Negative quantity means that the customer returned said item. The InvoiceID will have a 'C' character at the front
columns = ["Invoice", "StockCode", "Quantity", "InvoiceDate", "Price"]
df_clean = df.dropna(subset=columns).copy()

# Replace missing CustomerID with a Guest ID, one Guest per Invoice
missing = df_clean["Customer ID"].isna()   
df_clean["Customer ID"] = df_clean["Customer ID"].astype("Int64").astype(str)

guest_num = pd.factorize(df_clean.loc[missing, "Invoice"])[0]
df_clean.loc[missing, "Customer ID"] = "G" + pd.Series(guest_num, index=df_clean.index[missing]).astype(str).str.zfill(4)

df_removeChar = df_clean[
    (df_clean["StockCode"].str.len() >= 5)
    & (~df_clean["StockCode"].str[0].str.isalpha())
    & (df_clean["Quantity"] != 0)
    & (df_clean["Price"] > 0)
].drop_duplicates(subset=["Invoice", "StockCode", "Quantity"])

df_removeChar.to_csv("clean4.csv", index=False)

In [4]:
# Rows that were removed (same conditions as above, evaluated on the same frame so the indexes line up)
stock = df_clean["StockCode"].astype(str)
columns = ["Invoice", "StockCode", "Quantity", "InvoiceDate", "Price"]

corruptData = df_clean[(df_clean[columns].isna().any(axis=1))
                 | (df_clean.duplicated(subset=["Invoice", "StockCode", "Quantity"]))
                 | (df_clean["Price"] <= 0)
                 | (stock.str.len() <= 4)
                 | (stock.str[0].str.isalpha())
                 | (df_clean["Quantity"] == 0)]
corruptData.to_csv("corruptData2.csv", index=False)

In [5]:
#Temporary, seeing where is G0001
print(df_clean[df_clean["Customer ID"].isin(["G0000","G0001","G0002","G0003"])][["Invoice","StockCode","Quantity","Price"]])

    Invoice StockCode  Quantity  Price
263  489464     21733       -96    0.0
283  489463     71477      -240    0.0
284  489467    85123A      -192    0.0
470  489521     21646       -50    0.0
